# Replay Buffer Storage in Continual Learning of CLIP**Alexie Linardatos**  Progress update, phase two. 11-task MTIL, CLIP ViT-B/16, single seed.### Status in one paragraphPhase one (ExRD) stores 11,000 images in its replay buffer, which is 6.6 GB, a cost no MTIL paperreports. Phase two asks what that buffer actually needs to contain. I built two alternatives:feature replay, which stores 512-d embeddings instead of images at 588x less memory, and a port ofREMIND, which stores compressed mid-network activations and prevents drift by freezing the lowerhalf of the encoder. Both land about 1.4 points below ExRD on Last. Two thirds of that loss is asingle task. The drift-correction methods I built to fix it did not work, and I can now show whythe compression approach cannot work either. The results are below, followed by four questions Ineed answered about scope.

## 1. Why store embeddings rather than imagesReading the objective, the loss terms route asymmetrically across CLIP's two towers. $L_{zscl}$ and$L_{RD}$ both compute the teacher's text embeddings under no-grad, so they reach the image encoderonly. The replay cross-entropy recomputes class-name embeddings with the live model, making it theonly term in the whole objective that updates the text encoder for previously seen classes.| Loss term | Image encoder | Text encoder (old classes) ||---|---|---|| $L_{zscl}$ | yes | no || $L_{RD}$ | yes | no || $L_{2}$ | no | no || $L_{rep}$ (replay) | yes | yes |So the image encoder has three constraints and the text encoder has one. Replay's image-sidegradient is largely redundant; its text-side gradient is unique. Storing images is how you buy theredundant half. That is the argument for storing a 512-d embedding instead, and it is the part ofthis work I have not found anywhere in the literature.Note: the text_loss flag transposes the logits. It does not route gradient to the text encoder.This is easy to misread.

## 2. What I built**Feature replay (v0).** At the end of each task, encode its exemplars once and store 512-d fp16vectors. Discard the images. The replay logits become a stored constant times the live textembeddings, so only the text encoder receives gradient.**REMIND port.** Store product-quantized activations from block 6 of 12 instead of the finalembedding, so replayed samples still flow through the upper blocks and carry a real gradient. Freezeeverything below block 6 after task 0, so the layers that produced the stored activations never moveand the activations cannot go stale.Both are flags on the existing trainer, so pixel runs stay bit-identical to ExRD.| | Bytes per exemplar | Buffer at 11k ||---|---|---|| Pixel replay (ExRD) | 602 kB | 6.62 GB || REMIND, m=32 | 6.3 kB | 73.7 MB || Feature replay | 1.02 kB | 11.3 MB |Two consequences of dropping the images, both of which became findings. Stored vectors cannot beaugmented, so the model sees the identical signal every replay step. And $L_{RD}$ needs real pixels,so it was switched to current-task images.

## 3. Results| Run | Avg | Last | T[IN] | T[UT] | Storage ||---|---|---|---|---|---|| ExRD / v4 (pixel replay) | 77.17 | 86.17 | 68.37 | 69.18 | 6.6 GB || featrep v0 (pure) | 76.08 | 84.79 | 68.12 | 68.81 | 11 MB || featrep lp_both | 76.23 | 84.89 | 67.95 | 68.99 | 11 MB || featrep lp_text | 76.06 | 84.73 | 68.05 | 68.78 | 11 MB || featrep sdc_image | 76.20 | 84.65 | 67.90 | 68.88 | 11 MB || REMIND l6 m32 | 76.00 | 84.62 | 68.55 | 68.88 | 69 MB |T[IN] is the held-out ImageNet column, our internal convention. T[UT] is the upper triangle, theconvention ZSCL, DIKI, BCL and AFA report. External comparisons use T[UT].| External method | Avg | Last | Transfer | Buffer ||---|---|---|---|---|| ZSCL (ICCV 2023) | 75.4 | 83.6 | 68.1 | not reported || DIKI | 76.3 | 85.1 | 68.3 | none || BCL | 76.7 | 85.0 | 68.9 | none || AFA (2025) | 78.5 | 87.2 | 70.3 | none |

## 4. Four findings**The penalty is one task, and it scales with task age.** Comparing final-row accuracies of ExRDagainst feature replay, eight of eleven tasks are within noise and two are slightly better. Aircraftis task 0, stored first and exposed to ten subsequent boundaries.| Task | Position | ExRD | featrep v0 | Delta ||---|---|---|---|---|| **Aircraft** | 0 | 54.52 | 44.55 | **-9.97** || CIFAR100 | 2 | 81.98 | 79.74 | -2.24 || DTD | 3 | 76.06 | 74.73 | -1.33 || StanfordCars | 9 | 85.54 | 84.63 | -0.91 || EuroSAT, MNIST | 4, 7 | | | positive || other five | | | | within 0.81 || **Mean (Last)** | | **86.17** | **84.79** | **-1.38** |Aircraft alone is 65.8% of the loss. Aircraft plus CIFAR100 is 80.6%.**Drift correction does not recover it.** I built a ladder of stale-feature adaptation methods (SDC,label propagation, learned maps) crossed with image, text and both anchors. The text-anchor arm wasmeant to be the novel contribution, since a class's text embedding is exactly recomputable with nostored image. All four variants land within 0.24 points of each other on Last. On Aircraft theordering was image > both > text, with text roughly equal to no adaptation. The likely cause isCLIP's modality gap. Reported as a negative result, and partly anticipated by SeGP-CL (2026).**Drift prevention works better than correction, but costs plasticity.** REMIND's freeze recoversAircraft from 44.55 to 49.02, more than any adaptation method managed. It pays for it onStanfordCars, down 2.70, and EuroSAT, down 2.02, so the mean does not move.**Replay fidelity appears irrelevant.** REMIND matched feature replay while replaying activationscorrupted at 58% relative reconstruction error. If fidelity mattered, that could not have happened.This follows directly from the routing argument in section 1: if the image-side gradient isredundant, it barely matters what image produces it.

## 5. Why REMIND, and why its compression cannot be fixedSection 4 produces two joint requirements: prevent drift rather than correct it, since correctionwas measured and found weak, and keep the replayed sample on a live gradient path, since a storedfinal embedding gives up the image tower entirely. Every alternative family fails one of the two.| Family | Example | Prevents drift | Live gradient ||---|---|---|---|| Drift compensation | SDC, A2LP, Iscen | no, corrects after the fact | no || Generative latent replay | CGIL (BMVC 2024) | partly, the VAE itself drifts | no || Generative pixel replay | LoRA-Loop | yes | yes, but needs a generator and a pixel pipeline || Rehearsal-free | AFA, DIKI, BCL | not applicable, no buffer | not applicable || Compressed activation replay | **REMIND** | **yes, by freezing** | **yes, mid-network tap** |ACAE-REMIND and SHARP are refinements of REMIND rather than alternatives, so implementing REMINDimplements the family's common core and turns those two into follow-up ablations.**But REMIND's compression ratio was never transferable.** Each PQ subvector holds $D/m$ dimensionsin 8 bits, so rate per dimension is $8m/D$ and the relative error floor is $2^{-R}$.| $m$ | Bits per dim | Error floor | Buffer at 11k ||---|---|---|---|| 32 | 0.33 | 0.794 | 69 MB || 64 | 0.67 | 0.630 | 139 MB || 128 | 1.33 | 0.397 | 277 MB || 256 | 2.67 | 0.157 | 555 MB |Observed errors at m=32 were 0.765, 0.740 and 0.487 against a floor of 0.794. The codebook isalready at the theoretical limit for its bit budget. This is not a tuning problem, and a learnedrotation (OPQ) cannot beat a rate bound either. REMIND worked because post-ReLU CNN feature maps aresparse and low-rank; ViT residual streams are far closer to full-rank, so the same bits buy muchless fidelity.I did fix two real CLIP mismatches in the port: the codebook was being fitted once on Aircraft andreused for MNIST and EuroSAT, now one codebook per task; and transformer residual streams haveoutlier channels that swamp contiguous PQ subspaces, now standardised per channel. Both are running.

## 6. Caveats I want on the record**A bug makes the adaptation ladder not cleanly measured.** Task 0 accuracy must be identical acrossruns, since the buffer is empty. ExRD and REMIND reproduce each other exactly at 52.48. All fourfeature-storage runs diverge, spread 1.20 points, tracking flags not read until task 1. No seeddiffers, so this is a bug in the feature-storage path. The ladder's 0.24 point spread is smallerthan the artefact, so that null result needs re-running. The useful corollary is that the pipelineis otherwise deterministic, so single-seed comparisons are valid and no seed sweep is needed.**One design concession was never measured.** Moving $L_{RD}$ to current-task images was validatedon Transfer only, costing about 0.06. Its effect on Last was never measured, and it could accountfor a meaningful share of the 1.38 point gap.

## 7. Where this stands, and what I need decided### The case forA well-posed question asked where nobody has asked it, a mechanism rather than only a number, twointerventions that behave as the mechanism predicts, honest negative results with explanations, andit builds directly on phase one.### The case againstNo state of the art. AFA beats us on all three metrics with zero exemplars. The headline noveltyclaim, text anchors, failed. The storage axis is undercut by rehearsal-free methods with no bufferat all.### Questions1. Is characterisation plus mechanism plus an honest negative result sufficient, or does the thesis   need a method that wins a number?2. Is the storage axis defensible given rehearsal-free state of the art? My position is that the   comparison is fair within replay methods, and that AFA's router, which infers task identity at   test time, is an easier problem setting. Is that framing acceptable?3. Is the gradient-routing analysis in section 1 strong enough to be the primary contribution, with   the empirical work as its validation?4. Worth pursuing depth-stratified replay? Store older tasks at a deeper layer so fewer trainable   blocks sit above them, giving drift protection proportional to exposure with no global freeze. It   costs nothing in storage, since token count is constant across ViT depth, and it sacrifices only   the image-side gradient section 1 argues is redundant.### Remaining work| | Task | Cost ||---|---|---|| 1 | Fix the task-0 divergence bug | hours, no GPU || 2 | Measure the $L_{RD}$ relocation on Last | 1 run || 3 | PQ sweep over m in 32, 64, 128, to test whether fidelity matters | 3 runs, parallel || 4 | Re-run the adaptation ladder after the fix | 2 to 3 runs || 5 | Optional: depth-stratified replay | 1 day plus 3 runs |About 4 to 12 GPU-days, parallelisable, plus writing. Item 3 is the most informative: if Last isflat across a 2x range of reconstruction error, that demonstrates replay fidelity is irrelevantunder this objective, which would tie the thesis together with a result rather than a caveat.